## Load and clean data

We have done the followings to clean the dataframe:

1. Remove duplicate rows
    - duplicates were generated if a shared expression was used for multiple Fribbles. One row contains necessary information about Fribbles, rounds, speakers, etc so removing duplicates is essential to avoid counting multiple instances of alignment when there's supposed be only one alignment.
1. Remove labels that only contained: 
    - position-related info (*linkerkant*)
    - hedges (*sort of*)
    - numbers used as adjective (*zitten twee*)
    - modifier (*very*)
    - discourse markers not related to Fribbles (*okay*, *anyway*, *again*)

In [7]:
import pandas as pd
import numpy as np
from ast import literal_eval
import os
import deepl
from speach import elan
from tqdm import tqdm
from datetime import datetime, timedelta
import time
import spacy
import re

# load the spacy model
nlp = spacy.load("nl_core_news_lg")

# DeepL Preparation
with open("P:/workspaces/mld-akamine/working_data/deepl_authkey.txt", "r") as f:
    auth_key = f.read().strip()
global translator
translator = deepl.Translator(auth_key)

# paths
dataset = "zoom" # "zoom" or "small"
folder_name = "zoom" if dataset == "zoom" else "cabb_small"
data_f = f"../data/{folder_name}/"
elan_f = f"../data/{folder_name}/elan/"

### Extract unique shared labels (will be used to remove unwanted labels in cleaning)

In [2]:
def translate_labels(df, source_lang, target_lang):
    ### Translate text
    column_name = 'label'
    eng_column_name = column_name + '_eng'
    df[eng_column_name] = translator.translate_text(
        df[column_name], source_lang=source_lang, target_lang=target_lang, split_sentences=0
        )


output_filename = '../data/words_lists/labels.csv'
if not os.path.exists(output_filename):
    ### Load the labels from the zoom dataset
    df_shared_labels_zoom = pd.read_csv('../data/zoom/exp_info.csv')
    unique_labels_zoom = df_shared_labels_zoom['label'].unique()
    df_unique_labels = pd.DataFrame(unique_labels_zoom, columns=['label'])
    ### Load the labels from the small dataset
    df_shared_labels_small = pd.read_csv('../data/cabb_small/exp_info.csv')
    unique_labels_small = df_shared_labels_small['label'].unique()
    df_unique_labels_small = pd.DataFrame(unique_labels_small, columns=['label'])
    ### Combine the labels from both datasets
    df_unique_labels = pd.concat([df_unique_labels, df_unique_labels_small])
    ### Translate labels
    translate_labels(df_unique_labels, 'NL', 'EN-US')
    ### fill in the remove column based on the criteria of the previous study (Akamine et al., 2024) & new criteria
    df_unique_label_previous = pd.read_csv('../data/words_lists/labels_akamine_2024.csv')
    remove_labels = df_unique_label_previous[df_unique_label_previous['remove'] == 'x']['label'].to_list()
    remove_labels = [x.lower() for x in remove_labels]
    # remove_labels.extend(['ja', 'oh ja', 'ja oké', 'ja precies', 'ja top', 'ehm', 'yes', 'eh', 'oh ik hebben', 'eh ik hebben', 'oh', 'uh',
    #                       'een beetje', 'precies', 'ik hebben één', 'één', 'aan de linker en', 'eh deze hebben', 'deze staan op', 'hebben',
    #                       'daar links', 'dan aan de linkerkant', 'en dan links', 'eh en dan', 'mhm', 'lijken een', 'op de bovenkant',
    #                       'aan allebei de kant', 'hebben twee', 'daarboven', 'eh deze', 'niet echt', 'eh deze', 'hoi', 'bovenop', 'naar links',
    #                       'erin', 'er bovenop', "eh zo'n", 'een beetje op', 'oeh', 'de achterkant', 'ik hebben eh', 'lijken maar', 'daarachter',
    #                       'eigenlijk', 'hm', 'eraan', 'linker', 'je bedoelen', "ook zo'n", 'net hebben', 'op staat', 'oh ik hebben nu',
    #                       'ehm ja', 'daaruit', 'en dan twee', 'waarvan ik zeggen', 'eentje aan de', 'ook', 'eh de', 'waarvan', 'euh', 'staan ook',
    #                       'links twee', 'nog een', 'en dan bovenin', 'en dan onderin', 'daarnaast', 'de linker', 'ehm deze hebben', 'eh ik hebben die',
    #                       'ik noemen', 'beschrijven', 'waar die', 'en dan', 'aan de rechtervoorzijde', 'ehm ik hebben er nu één', 'lijken alsof',
    #                       'nu één', 'op zijn kant', "zo'n", 'euh ik hebben', 'euh ik hebben nu die', 'rechtop', 'ohja', 'er twee', 'letter',
    #                       'links één naar rechts', 'er één', 'er', 'en daarnaast', 'met het euh', 'ja even', 'de linker zijn', 'euhm',
    #                       'oke', 'even kijken waar zijn hij', 'aan beide kant', 'ik vinden het een beetje op', 'alleen', 'en aan de rechterkant',
    #                       'ook weer', 'eh even kijken', 'zijn weer', 'ik vinden het', 'ook twee', 'er bijna', 'bijna', 'omhoog', 'erin zijn',
    #                       'maar twee', 'dit zijn weer', 'soort', 'euh deze hebben', 'lijken', 'even', 'uhm', 'kijken', 'top', 'ehh', 'eeh',
    #                       'link', 'object', 'wacht', 'eerst', 'even zoeken', 'echt', 'zoeken', 'misschien', 'staan', 'helemaal', 'met een soort',
    #                       'weer die', 'dit zijn weer die', 'vinden ik', 'ik vinden', 'niet', 'net', 'ding', 'onderkant van', 'een naam voor',
    #                       'o', 'k', 'a', 'zijn eigenlijk', 'rechts hebben hij', 'ik hebben er één' '(laugh)', 'laugh', 'een heel', 'wachten',
    #                       'dus één', 'een recht', 'beginnen en', 'ik hebben er één'])
    remove_labels.extend(['ja', 'oh', 'oké', 'precies', 'top', 'ehm', 'yes', 'eh', 'ik', 'hebben', 'één', 'uh',
                          'aan', 'de', 'linker', 'linkerkant', 'en', 'dan', 'links', 'mhm', 'lijken', 'een', 'beetje',
                          'op', 'deze', 'staan', 'allebei', 'kant', 'twee', 'daarboven', 'niet', 'echt', 'euh',
                          'hoi', 'bovenop', 'naar', 'erin', 'er', 'bovenop', "zo'n", 'oeh', 'achterkant',
                          'eigenlijk', 'hm', 'eraan', 'je', 'bedoelen', 'ook', 'net', 'nu', 'waarvan', 'dit',
                          'zeggen', 'eentje', 'waar', 'die', 'rechtervoorzijde', 'rechtop', 'ohja', 'letter',
                          'met', 'het', 'even', 'zijn', 'euhm', 'oke', 'kijken', 'weer', 'alleen', 'beide',
                          'vinden', 'het', 'soort', 'uhm', 'ehh', 'eeh', 'link', 'object', 'wacht', 'eerst',
                          'zoeken', 'echt', 'helemaal', 'misschien', 'wachten', 'dus', 'recht', 'beginnen',
                          '(laugh)', 'laugh', 'heel', 'wachten', 'een', 'recht', 'ik hebben er één', 'naam', 'voor',
                          'daarachter', 'maar', 'nog', 'bovenin', 'onderin', 'daarnaast', 'beschrijven', 'noemen',
                          'alsof', 'hij', 'bijna', 'omhoog', 'o', 'k', 'a', 'm', 'rechts', 'ding', 'onderkant', 'van'])
    # remove listed texts from the labels
    df_unique_labels['label_clean'] = df_unique_labels['label'].str.lower().apply(lambda x: re.sub(r'\b(?:' + '|'.join(remove_labels) + r')\b', '', x))
    df_unique_labels['label_clean'] = df_unique_labels['label_clean'].str.strip()
    df_unique_labels['remove'] = np.where(df_unique_labels['label_clean'] == '', 'x', '')
    df_unique_labels.loc[df_unique_labels['label'].str.lower().isin(remove_labels), 'remove'] = 'x'
    df_unique_labels.to_csv(output_filename, index=False)

In [8]:
### Load the data
df_shared_labels = pd.read_csv(data_f + 'exp_info.csv')
df_all_unique_labels = pd.read_csv('../data/words_lists/labels.csv')

### remove duplicate rows as the dataframe contains multiple rows for each label 
### if the shared expression was used for multiple Fribbles (e.g., 4 rows if the label was used for 4 Fribbles)
columns = df_shared_labels.columns.to_list()
columns = columns[2:28]
df_shared_labels = df_shared_labels.drop_duplicates(subset=columns, keep='first')
print("before removing labels: ", df_shared_labels.shape)

### remove labels based on df_remove_labels
# select rows if remove column is x
df_remove_labels = df_all_unique_labels[df_all_unique_labels['remove'] == 'x']
# remove rows with labels that are in df_remove_labels
df_shared_labels = df_shared_labels[~df_shared_labels['label'].isin(df_remove_labels['label'])]
print("after removing labels: ", df_shared_labels.shape)

### Save the data
df_shared_labels.to_csv(data_f + 'exp_info_cleaned.csv', index=False)

before removing labels:  (2945, 40)
after removing labels:  (1776, 40)


## Extract all turns from elan files
We extract all turns from elan files and add information about the turn such as word count and duration.

In [25]:
def convert_time_string_to_float(str_timestamp):
    t = datetime.strptime(str_timestamp, "%H:%M:%S.%f")
    t_formatted = timedelta(hours=t.hour, minutes=t.minute, seconds=t.second, microseconds=t.microsecond).total_seconds()
    return t_formatted

def retrieve_annots_from_elan(speech_tier, speaker):
    array = np.array([])
    for ann in speech_tier:
        from_ts = ann.from_ts.ts
        to_ts = ann.to_ts.ts
        from_ts_s = convert_time_string_to_float(from_ts)
        to_ts_s = convert_time_string_to_float(to_ts)
        text = ann.text
        array = np.append(array, [speaker, from_ts, to_ts, from_ts_s, to_ts_s, text])

def clean_text(text):
    ### remove extra spaces
    text = text.strip()
    ### remove non-lexical tokens such as #laugh#, <name>, /zjoev/, and (laugh)
    text = re.sub("\w*-", "", text)             # remove words followed by - e.g. word-
    text = re.sub("#\w*\s*#", "", text)            # remove hashtags with one words inside (e.g. #word#)
    text = re.sub("#[?+\s*]#", "", text)        # remove hashtags with question marks (e.g. #?#)
    text = re.sub("#\w+ \w+\s*#", "", text)     # remove hashtags with two words inside (e.g. #word word#)
    text = re.sub("#\w+ \w+ \w+\s*#", "", text) # remove hashtags with three words inside (e.g. #word word word#)
    text = re.sub("#\w+ \w+ \w+ \w+\s*#", "", text) # remove hashtags with four words inside (e.g. #word word word word#)
    text = re.sub("#\w+ \w+ \w+ \w+ \w+\s*#", "", text) # remove hashtags with five words inside (e.g. #word word word word word#)
    text = re.sub("<\w*\s*>", "", text)         # remove tags with one word inside (e.g. <name>)
    text = re.sub("/\w*\s*/", "", text)         # remove slashes with one word inside (e.g. /zjoev/)
    text = re.sub("\(\s*\)", "", text)          # remove parentheses with no words inside or space only
    # text = re.sub("\(\w*\)", "", text)        # remove parentheses with words inside
    # text = re.sub("\(\w*\s*\)", "", text)       # remove parentheses with words inside (e.g. (word), (word ))
    # text = re.sub("\(\w+ \w+\s*\)", "", text)   # remove parentheses with two words inside (e.g. (word word))
    # text = re.sub("\(\w+ \w+ \w+\s*\)", "", text)  # remove parentheses with three words inside (e.g. (word word word))
    # text = re.sub("\(\w+ \w+ \w+ \w+\s*\)", "", text)  # remove parentheses with three words inside (e.g. (word word word))
    # text = re.sub("\(\w+ \w+ \w+ \w+ \w+\s*\)", "", text)  # remove parentheses with three words inside (e.g. (word word word))
    # text = re.sub("\([?+\s*]\)", "", text)      # remove parentheses with question marks (e.g. (?))
    ### replace encoding issues
    text = text.replace("Ã©", "é")
    ### remove some special characters
    text = text.replace("*", "")
    text = text.replace("!", "")
    text = text.replace("?", "")
    text = text.replace("...", "")
    text = text.replace(",", "")
    text = text.replace(".", "")
    text = text.replace("(", "")
    text = text.replace(")", "")
    ### fix the typos
    text = text.replace("paddelstoel", "paddenstoel") 
    ### remove extra spaces again
    text = text.strip()

    return text


def calculate_n_words(text, lemma, pos_list):
    # count the number of words in the speech (except when the row only contains parentheses)
    # num_words = len(text.split()) if text != "(" or text != ")" or text != "()" else 0
    num_words = len(lemma.split())
    # # count the number of tokens in the speech (except parentheses)
    # num_tokens = len(nlp(text)) if text != "(" or text != ")" or text != "()" else 0
    # count the number of content words in the speech
    pos_func_words = ['DET', 'PRON', 'ADP', 'CCONJ', 'SCONJ', 'AUX', 'PART', 'PUNCT', 'SYM', 'X', 'INTJ', 'NUM', 'SPACE']
    num_content_words = len([pos for pos in pos_list if pos not in pos_func_words])
    
    return num_words, num_content_words



### Create a dataframe with all turns
if os.path.exists(data_f + 'all_turns_original.csv'):
    print("all_turns_original.csv already exists")
else:
    speakers = ['A', 'B']
    speech_tier_name = '_speech' if dataset == "zoom" else '_po'
    array = []
    elan_files = [filename for filename in os.listdir(elan_f) if filename.endswith(".eaf")]
    for filename in tqdm(elan_files):
        if filename.endswith(".eaf"):
            pair = filename.split(".")[0]
            elan_file = elan.read_eaf(elan_f + filename)
            temp_array = []
            for speaker in speakers:
                speech_tier = elan_file[speaker+speech_tier_name]
                if dataset == "zoom":
                    eng_tier = elan_file[speaker+speech_tier_name+'_eng']
                trial_tier = elan_file['trial']
                for ann in speech_tier:
                    from_ts = ann.from_ts.ts
                    to_ts = ann.to_ts.ts
                    from_ts_s = convert_time_string_to_float(from_ts)
                    to_ts_s = convert_time_string_to_float(to_ts)
                    duration = to_ts_s - from_ts_s
                    
                    original_text = ann.text
                    text = clean_text(original_text)
                    if text == "":
                        continue
                    # lemmatize the text
                    lemma = " ".join([token.lemma_ for token in nlp(text)])
                    try:
                        pos = [token.pos_ for token in nlp(lemma[0].lower() + lemma[1:])]
                    except Exception as e:
                        print(f"Error in processing text: {text} for {pair}_{speaker}. Error: {e}")

                    # get the English translation of the text
                    if dataset == "zoom":
                        try:
                            eng_text = [eng.text for eng in eng_tier if eng.from_ts.ts == from_ts][0]
                        except:
                            eng_text = None
                            print(f"eng_text not found for {pair}_{speaker}: {from_ts}  {text}")
                    else:
                        # eng_text = translator.translate_text(
                        #     original_text, source_lang='NL', target_lang='EN-US').text
                        eng_text = None
                    # calculate the number of words, tokens, and content words
                    num_words, num_content_words = calculate_n_words(text, lemma, pos)
                    
                    ### get the trial number based on the from_ts. 
                    # The trial number is the trial number of the trial whose start is after the from_ts and the end is before the to_ts
                    trial_ann = [trial for trial in trial_tier.annotations if trial.to_ts.ts <= from_ts and to_ts <= trial.to_ts.ts] # trial_start <= annotation <= trial_end
                    # when the annotation overlaps with no trial, the trial that has the largest overlap with the annotation is selected
                    if len(trial_ann) == 0:
                        trial_ann_start = [trial for trial in trial_tier.annotations if trial.from_ts.ts <= from_ts and from_ts <= trial.to_ts.ts] # trial_start <= annot_start <= trial_end
                        trial_ann_end = [trial for trial in trial_tier.annotations if trial.from_ts.ts <= to_ts and to_ts <= trial.to_ts.ts] # trial_start <= annot_end <= trial_end
                        trials = list(dict.fromkeys(trial_ann_start + trial_ann_end)) # remove duplicates
                        if len(trials) > 1:
                            # when the annotation overlaps with multiple trials, the trial that has the largest overlap with the annotation is selected (e.g., 1_1)
                            #        |-----------------annot------------------|
                            # |----------trial 1_1-----------|    |----------trial 1_2-----------|  
                            #        |-----start_overlap-----|    |end_overlap|
                            trial_1_end = convert_time_string_to_float(trials[0].to_ts.ts)
                            trial_2_start = convert_time_string_to_float(trials[1].from_ts.ts)
                            start_overlap = abs(from_ts_s - trial_1_end)
                            end_overlap = abs(to_ts_s - trial_2_start)
                            trial_ann = [trials[0]] if start_overlap > end_overlap or "*" in text else [trials[1]]
                        else:
                            trial_ann = trials
                    trial = trial_ann[0].text.replace(".", "_") if len(trial_ann) > 0 else None
                    
                    # append the data to the array
                    temp_array.append([pair, speaker, trial, 
                                        from_ts, to_ts, from_ts_s, to_ts_s, duration, 
                                        original_text, text, lemma, pos, eng_text,
                                        num_words, num_content_words])
            array.extend(temp_array)

    ### Create a dataframe with all turns and save it
    df_all_turns = pd.DataFrame(array, columns=['pair', 'speaker', 'trial', 
                                                'from_ts', 'to_ts', 'start', 'end', 'duration', 
                                                'original_text', 'text', 'lemma', 'pos', 'eng_text',
                                                'num_words', 'num_content_words'])
    df_all_turns.sort_values(by=['pair', 'start'], inplace=True)
    df_all_turns.reset_index(drop=True, inplace=True)
    df_all_turns.insert(0, "turn_id", df_all_turns.index + 1)
    df_all_turns.to_csv(data_f + 'all_turns_original.csv', index=False)

100%|██████████| 45/45 [05:01<00:00,  6.71s/it]


## Reformat the dataframe
Here, we will reformat the dataframe by expanding each row based on the freq column. For example, if the label "vierkant" has 20 frequencies, then we will add 20 rows in a new dataframe each corresponding to each occurance.

In [9]:
df_shared_labels = pd.read_csv(data_f + 'exp_info_cleaned.csv')
df_all_turns = pd.read_csv(data_f + 'all_turns_original.csv')

##### make an empty dataframe with same columns as df_shared_labels
new_columns = ["pair", "int_pair", "label", "length", "target", 
               "n_fribbles", "speaker", "freq", "turn", "round", "trial",
               "from_ts", "to_ts", "start", "end", "duration",
               "pos_seq", "shared_expression", "n_shared_expressions",
               "turn_id"]
new_df = pd.DataFrame(columns = new_columns)

##### add rows to new_df
### iterate through each row in df_shared_labels
for index, row in tqdm(df_shared_labels.iterrows()):
    speakers = literal_eval(row['speakers'])
    fribbles= literal_eval(row['fribbles'])
    rounds = literal_eval(row['rounds'])
    from_ts = literal_eval(row['from_ts'])
    to_ts = literal_eval(row['to_ts'])
    turns = row['turns'][1:-1].split(" ") # we cannot use literal_eval for turns because it doesn't separate each turn by comma
    turns = [x for x in turns if x != ''] # remove empty string

    ### itearate through each element in the list of speakers column
    for i in range(len(speakers)):
        ### create a new row
        try:
          mask = (df_all_turns['pair'] == row['pair']) & (df_all_turns['speaker'] == speakers[i]) & (df_all_turns['start'] == from_ts[i]) & (df_all_turns['end'] == to_ts[i])
          trial = df_all_turns[mask]['trial'].values[0]
          start = df_all_turns[mask]['start'].values[0]
          end = df_all_turns[mask]['end'].values[0]
          duration = end - start
          turn_id = df_all_turns[mask]['turn_id'].values[0]
          new_row = pd.DataFrame([[row['pair'], row['int_pair'], row['label'], row['length'], fribbles[i], 
                                  row['#fribbles'], speakers[i], row['freq'], turns[i], rounds[i], trial,
                                  from_ts[i], to_ts[i], start, end, duration,
                                  row['pos_seq'], row['shared expressions'], row['#shared expressions'],
                                  turn_id]], 
                                columns = new_columns)
          ### append new row to new_df using concat
          new_df = pd.concat([new_df, new_row], ignore_index=True)
          
        except:
          print("Error: ", row['pair'], speakers[i], from_ts[i], to_ts[i])

### save new_df
new_df.to_csv(data_f + 'exp_info_cleaned_long.csv', index=False)

0it [00:00, ?it/s]C:\Users\shoaka\AppData\Local\Temp\ipykernel_19900\1343866687.py:40: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_df = pd.concat([new_df, new_row], ignore_index=True)
1776it [00:26, 68.01it/s]


## Extract lexical alignment
Next, we will extract lexical alignment from the shared_expressions_long.csv dataframe.

The definition of lexical alignment is **the repetition of verbal expressions (e.g., single word, word phrase) about the same Fribble that are produced by two different speakers**. This definition is consistent with gestural alignment.

Note that we only looked at nouns, verbs, and adjectives (or phrases containing a noun, verb or adjective).

#### Functions

In [10]:
def make_alignment1_df(row):
    df = row.to_frame().transpose() #this is to make sure that the dataframe is transposed so that the dataframe is added as a row
    df = df.drop(columns=['index', 'pair', "int_pair", 'label', 'length', 'flag',
                          'target', 'from_ts', 'to_ts', 'n_fribbles', 'freq', 
                          'pos_seq', 'shared_expression', 'n_shared_expressions'])
    df = df.add_suffix('_1')
    df = df.reset_index(drop=True) #this is to make sure that the index of the dataframe is reset to 0; the index of two dataframes that will be concatenated should be the same
    return df

def make_alignment2_df(row):
    df = row.to_frame().transpose()
    df = df.drop(columns=['index', 'pair', "int_pair", 'label', 'length', 'flag',
                          'target', 'from_ts', 'to_ts', 'n_fribbles', 'freq', 
                          'pos_seq', 'shared_expression', 'n_shared_expressions'])
    df = df.add_suffix('_2')
    df = df.reset_index(drop=True)
    return df

def append_to_alignment_df(main_df, df1, df2, referent_df):
    # concatenate the two dataframes as one row
    df = pd.concat([df1, df2], axis=1)
    cols = ['pair', 'target', 'label', 'length', 'freq', 'pos_seq', 
            'n_fribbles', 'shared_expression', 'n_shared_expressions']

    for col in cols:
        df[col] = referent_df[col]

    main_df = pd.concat([main_df, df], ignore_index=True)
    return main_df

#### Let's run the functions!

In [11]:
df_shared_expressions = pd.read_csv(data_f + 'exp_info_cleaned_long.csv')

pairs = df_shared_expressions['pair'].unique()

# make an empty pandas dataframe that will contain the alignment information
# the alignment dataframe will have the the following columns:
alignment_df = pd.DataFrame(columns=['pair', 'label', 'target', 
                                     'speaker_1', 'round_1', 'trial_1', 'turn_1', 
                                     'start_1', 'end_1', 'duration_1', 'turn_id_1',
                                     'speaker_2', 'round_2', 'trial_2', 'turn_2', 
                                     'start_2', 'end_2', 'duration_2', 'turn_id_2',
                                     'length', 'freq', 'pos_seq', 'n_fribbles', 
                                     'n_shared_expressions'])

for pair in tqdm(pairs):
    pair_df = df_shared_expressions.loc[df_shared_expressions['pair'] == pair]
    
    # get the list of target referents
    referents = list(pair_df['target'].unique())
    
    for referent in referents:
        labels = list(pair_df['label'].unique())

        for label in labels:
            # get the rows that have the same pair, label, and target
            referent_df = pair_df.loc[(pair_df['label'] == label) & (pair_df['target'] == referent)]
            referent_df = referent_df.sort_values(by=['turn']).reset_index()
            referent_df['flag'] = np.where((referent_df['speaker'] != referent_df['speaker'].shift(-1)) & (referent_df['speaker'].shift(-1).notnull()), 1, 0)

            # add the rows that have the flag set to 1 to alignment_df
            # the rows with the flag set to 1 are the rows where the gesture alignment initiates, and the next row is where the gesture alignment is established
            # the information of the rows with the flag set to 1 is added to the alignment_df under the columns that end with _1
            # the information of the next row is added to the alignment_df under the columns that end with _2
            alignment = False # alignment is set to True when the flag is 1: this will be used to add the information of the next row to the alignment_df

            for index, row in referent_df.iterrows():
                if alignment == True:
                    alignment2_df = make_alignment2_df(row)
                    alignment_df = append_to_alignment_df(alignment_df, alignment1_df, alignment2_df, referent_df)
                    alignment = False
                
                if row["flag"] == 1:
                    alignment1_df = make_alignment1_df(row)
                    alignment = True
                else:
                    alignment = False

alignment_df.to_csv(data_f + 'lexical_alignment.csv', index=False)

100%|██████████| 45/45 [01:03<00:00,  1.41s/it]
